In [1]:
from EarthquakeSignal import config, EarthquakeBatchProcessor
from LadrunoGraphStyle import set_default_plot_params
set_default_plot_params()

In [2]:
# pip list

### Set config manual 

In [3]:
config = {
    'file_extension': '.txt',
    'unit_factor': 1,
    
    'apply_baseline_correction': True,
    'apply_arias_analysis': True,
    'apply_fourier_analysis': True,
    '_compute_newmark_spectra': True,
    'compute_rotd': True,
    'print_summary': True,
    
    'plot_signals': True,
    'plot_corrected_signals': True,
    'plot_arias_signals': True,
    'plot_fourier_signals': True,  
    'plot_newmark_spectra': True,
    'plot_rotd': True,
    
    'writer': True, 
}


In [4]:
registers_path = r'data/Selected'


In [ ]:
processor = EarthquakeBatchProcessor(registers_path, config ,  n_jobs=4) 
earthquake = processor.process_all()


Processing earthquakes:  33%|███▎      | 4/12 [00:00<00:00, 31.32it/s]

### For each earthquake

In [ ]:
earthquake['APED'].print_summary()
earthquake['APED'].plot_original_signals()
earthquake['APED'].plot_corrected_signals()
earthquake['APED'].plot_arias_signals()
earthquake['APED'].plot_fourier_signals()
earthquake['APED'].plot_newmark_spectra()


In [ ]:
record = earthquake['APED']


print("🔹 Arias Intensity")
print(f"Significant Duration Start  : {record.arias['H1']['t_start']:.3f} s")
print(f"Significant Duration End    : {record.arias['H1']['t_end']:.3f} s")
print(f"Total Arias Intensity       : {record.arias['H1']['IA_total']:.5f} m/s")
print(f"Destructive Potential Index : {record.arias['H1']['pot_dest']:.5f}")


print("\n🔹 Fourier Analysis")
print("Dominant Periods (s):", record.fourier['H1']['dominant_periods'])

## Keys and Methods

In [ ]:

first_key = next(iter(earthquake))
eq = earthquake[first_key]

print("Keys:")
for key in vars(eq):
    print(f"  - {key}")

print("\nMethods:")
import inspect
methods = [name for name, obj in inspect.getmembers(eq, inspect.ismethod) if not name.startswith('_')]
for m in methods:
    print(f"  - {m}()")

In [ ]:
print("PGA Comparison (direct vs spectral) for each record and component:")
print(f"{'Record':<20} {'Comp':<4} {'PGA':>12}" )

for name, eq in earthquake.items():
    print("--"*10)
    for comp in ['H1', 'H2', 'V']:
        psa_array = eq.newmark_spectra[comp]['PSa']
        pga = psa_array[0]

        print(f"{name:<20} {comp:<4} {pga:12.4f}")
              

### All spectra

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# --- Calcular ruta correcta ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
output_dir = os.path.join(project_root, 'outputs')
os.makedirs(output_dir, exist_ok=True)

# Puedes cambiar esta ruta si quieres guardar en otro lugar
svg_path = r'C:\Users\ppala\OneDrive\01. Brain\10. Ph.D U ANDES\07. Publicaciones\01. AmplificacionMuros\02. Modelos Shell\04. Model10stories\SelectedSeismic'

# --- Graficar espectros ---
plt.figure(figsize=(14, 4))

# Subplot para H1
plt.subplot(1, 2, 1)
for rsn, eq in earthquake.items():
    try:
        T_H1 = eq.newmark_spectra['H1']['T']
        PSa_H1 = eq.newmark_spectra['H1']['PSa']
        plt.plot(T_H1, PSa_H1, linewidth=1.5, label=f"{rsn}")
    except KeyError:
        print(f"Warning: {rsn} missing H1 spectrum")

plt.title("Component H1", fontweight='bold', fontsize=12)
plt.xlabel("Period [s]", fontweight='bold', fontsize=12)
plt.ylabel("PSa [g]", fontweight='bold', fontsize=12)
plt.grid(True)
plt.xlim(left=0, right=5)
plt.tick_params(axis='both', labelsize=12)
plt.legend(loc='upper right', ncol=2, fontsize=12)

# Subplot para H2
plt.subplot(1, 2, 2)
for rsn, eq in earthquake.items():
    try:
        T_H2 = eq.newmark_spectra['H2']['T']
        PSa_H2 = eq.newmark_spectra['H2']['PSa']
        plt.plot(T_H2, PSa_H2, linewidth=1.5, label=f"{rsn}")
    except KeyError:
        print(f"Warning: {rsn} missing H2 spectrum")

plt.title("Component H2", fontweight='bold', fontsize=12)
plt.xlabel("Period [s]", fontweight='bold', fontsize=12)
plt.ylabel("PSa [g]", fontweight='bold', fontsize=12)
plt.grid(True)
plt.xlim(left=0, right=5)
plt.tick_params(axis='both', labelsize=12)
plt.legend(loc='upper right', ncol=2, fontsize=12)

plt.suptitle("All Records", fontsize=12, fontweight='bold')

# --- Guardar figuras ---
plt.savefig(os.path.join(svg_path, 'seismic_selected.pdf'), format='pdf', bbox_inches='tight')
plt.savefig(os.path.join(svg_path, 'seismic_selected.svg'), format='svg', bbox_inches='tight')
plt.show()


### Rotd100

In [ ]:


# Create figure with 1 row and 1 column
plt.figure(figsize=(7, 4))

# Subplot for ROTD100
plt.subplot(1, 1, 1)
for rsn, eq in earthquake.items():
    try:

        T = eq.rotd['T']
        PSa_rotd100 = eq.rotd['ROTD100']
        pga = round(PSa_rotd100[0], 3)
        plt.plot(T, PSa_rotd100, linewidth=1.5, label=f"{rsn} -PGA:{pga} g")
    except KeyError:
        print(f"Warning: {rsn} missing ROTD100 spectrum")

plt.title("ROTD100 - All Records", fontsize=10, fontweight='bold')
plt.xlabel("Period [s]", fontsize=9, fontweight='bold')
plt.ylabel("PSa [g]", fontsize=9, fontweight='bold')
plt.grid(True, which='both')
plt.xlim(left=0, right=5)
plt.tick_params(axis='both', labelsize=8)
plt.legend(fontsize=7, loc='upper right', ncol=2)
plt.tight_layout()
plt.show()


In [ ]:


# Create figure with 1 row and 1 column
plt.figure(figsize=(7, 4))

# Subplot for ROTD50
plt.subplot(1, 1, 1)
for rsn, eq in earthquake.items():
    try:

        T = eq.rotd['T']
        PSa_rotd50 = eq.rotd['ROTD50']
        pga = round(PSa_rotd50[0], 3)
        plt.plot(T, PSa_rotd50, linewidth=1.5, label=f"{rsn} -PGA:{pga} g")
    except KeyError:
        print(f"Warning: {rsn} missing ROTD50 spectrum")

plt.title("ROTD50 - All Records", fontsize=10, fontweight='bold')
plt.xlabel("Period [s]", fontsize=9, fontweight='bold')
plt.ylabel("PSa [g]", fontsize=9, fontweight='bold')
plt.grid(True, which='both')
plt.xlim(left=0, right=5)
plt.tick_params(axis='both', labelsize=8)
plt.legend(fontsize=7, loc='upper right', ncol=2)
plt.tight_layout()
plt.show()


### Data Extraction

In [ ]:
import pandas as pd
import numpy as np

rows = []
for rsn, eq in earthquake.items():
    name = getattr(eq, "name", str(rsn))
    dt = getattr(eq, "dt", None)
    first_signal = next(iter(eq.signals.values()))
    n_samples = len(first_signal)
    total_duration = n_samples * dt

    H1_file = eq.component_names.get('H1', None) 
    H2_file = eq.component_names.get('H2', None) 
    V_file  = eq.component_names.get('V', None)

    # Significant Duration (SD) and Arias parameters
    H1_SD = eq.arias['H1']['t_end'] - eq.arias['H1']['t_start']
    H1_IA_total = eq.arias['H1']['IA_total']
    H1_pot_dest = eq.arias['H1']['pot_dest']

    H2_SD = eq.arias['H2']['t_end'] - eq.arias['H2']['t_start']
    H2_IA_total = eq.arias['H2']['IA_total']
    H2_pot_dest = eq.arias['H2']['pot_dest']

    V_SD = eq.arias['V']['t_end'] - eq.arias['V']['t_start']
    V_IA_total = eq.arias['V']['IA_total']
    V_pot_dest = eq.arias['V']['pot_dest']

    # First dominant peak
    H1_peak1 = eq.fourier['H1']['dominant_peaks'][0]
    H2_peak1 = eq.fourier['H2']['dominant_peaks'][0]
    V_peak1  = eq.fourier['V']['dominant_peaks'][0]

    # PGA = first value of PSa_corr
    H1_PGA = eq.newmark_spectra['H1']['PSa_corr'][0]
    H2_PGA = eq.newmark_spectra['H2']['PSa_corr'][0]
    V_PGA  = eq.newmark_spectra['V']['PSa_corr'][0]

    rows.append({
        "record_id": str(rsn),
        "name": name,
        "dt": dt,
        "n_samples": n_samples,
        "total_duration": total_duration,
        "H1_file": H1_file, "H1_SD": H1_SD, "H1_IA_total": H1_IA_total, "H1_pot_dest": H1_pot_dest*100, "H1_peak1": H1_peak1, "H1_PGA": H1_PGA,
        "H2_file": H2_file, "H2_SD": H2_SD, "H2_IA_total": H2_IA_total, "H2_pot_dest": H2_pot_dest*100, "H2_peak1": H2_peak1, "H2_PGA": H2_PGA,
        "V_file":  V_file,  "V_SD": V_SD,  "V_IA_total": V_IA_total,  "V_pot_dest": V_pot_dest*100,  "V_peak1": V_peak1,  "V_PGA": V_PGA
    })

df_records = pd.DataFrame(rows).sort_values("record_id").reset_index(drop=True)

print("✅ Registros encontrados:", len(df_records))
print(df_records.head(10))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# X positions
x = np.arange(len(df_records))
bar_width = 0.20

plt.figure(figsize=(12, 5))

# Grouped bars: H1, H2, V
plt.bar(x - bar_width, df_records["H1_PGA"], width=bar_width, label="H1")
plt.bar(x,              df_records["H2_PGA"], width=bar_width, label="H2")
plt.bar(x + bar_width,  df_records["V_PGA"],  width=bar_width, label="V")

plt.title("PGA by Component (H1 / H2 / V)", fontweight="bold")
plt.ylabel("PGA [g]", fontweight="bold")
plt.xticks(x, df_records["name"], rotation=90, ha="right")
plt.grid(axis="y")
plt.legend()
plt.tight_layout()
plt.show()
